# Principal component analysis

Dataset: https://huggingface.co/datasets/mnemoraorg/wisconsin-breast-cancer-diagnostic 

We need to run PCA on this dataset and understand which features are most important.

In [1]:
import numpy as np
import pandas as pd

import torch as t

from datasets import load_dataset
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path

# Site color palette (matches the blog theme)
SITE = dict(
    bg_primary   = '#213636',
    bg_secondary = '#1a2d2d',
    bg_tertiary  = '#2a4444',
    border       = '#3a5454',
    text_primary = '#c8c4b8',
    text_secondary = '#a09890',
    accent       = '#bdb76b',
    olive        = '#828631',
)

PLOTS_DIR = Path("./plots")
PLOTS_DIR.mkdir(exist_ok=True)

/Users/manavdahra/workspace/manavdahra.github.io/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = load_dataset("mnemoraorg/wisconsin-breast-cancer-diagnostic", split="train")

def map_label(x):
    if x["diagnosis"] == "M":
        x["diagnosis"] = 1
    else:
        x["diagnosis"] = 0
    return x

ds = ds.remove_columns(["id", "Unnamed: 32"])
ds = ds.train_test_split(test_size=0.1).map(map_label)

feature_cols = [c for c in ds["train"].column_names if c != "diagnosis"]
train_matrix = ds["train"].to_pandas()[feature_cols].to_numpy(dtype=np.float64)
mean = train_matrix.mean(axis=0)
std = train_matrix.std(axis=0)
std[std == 0] = 1.0  # guard against constant features

def normalize(batch: dict) -> dict:
    vals = np.column_stack([np.array(batch[c], dtype=np.float64) for c in feature_cols])
    normed = (vals - mean) / std
    return {f"norm_{c}": normed[:, i] for i, c in enumerate(feature_cols)}

ds = ds.map(normalize, batched=True, load_from_cache_file=False)

norm_cols = [f"norm_{c}" for c in feature_cols]

# Collect the normalized training data as a matrix (N x 30) and labels
X_all = np.column_stack([np.array(ds["train"][c], dtype=np.float64) for c in norm_cols])
y_all = np.array(ds["train"]["diagnosis"])

print(f"Data matrix: {X_all.shape[0]} samples x {X_all.shape[1]} features")
print(f"Class balance: {y_all.sum():.0f} malignant / {len(y_all) - y_all.sum():.0f} benign")

Map: 100%|██████████| 57/57 [00:00<00:00, 7231.56 examples/s]


Data matrix: 512 samples x 30 features
Class balance: 190 malignant / 322 benign


## PCA from scratch

Following the derivation in the blog post:

1. Center the data: $Z = X - \mu$
2. Compute the covariance matrix $\Sigma = \frac{1}{n} Z^T Z$
3. Eigendecompose $\Sigma$ and sort eigenvectors by decreasing eigenvalue
4. Take the top $k$ eigenvectors as columns of $U$
5. Project: $X' = ZU$

We do it via SVD of the centered data ($Z = V D U^T$, with $\lambda_j = d_j^2/n$),
which is what `np.linalg.svd` gives us directly and is numerically more stable
than forming $\Sigma$ explicitly.

In [4]:
# 1. Center
centered = X_all - X_all.mean(axis=0)

# 2 & 3. SVD of the centered data: Z = U_svd @ diag(S) @ Vt
#   The right singular vectors Vt are the eigenvectors of (1/n) Z^T Z,
#   and the eigenvalues are lambda_j = S_j^2 / n.
U_svd, S, Vt = np.linalg.svd(centered, full_matrices=False)
eigenvalues = S**2 / len(centered)

# Sort check: np.linalg.svd returns singular values in decreasing order already
print("Eigenvalues (top 5):", np.round(eigenvalues[:5], 3))
print(f"Total variance tr(Sigma) = {eigenvalues.sum():.3f}")
print(f"Trace of raw per-feature variances = {np.var(centered, axis=0, ddof=0).sum():.3f}")

Eigenvalues (top 5): [13.163  5.713  2.884  1.99   1.659]
Total variance tr(Sigma) = 30.000
Trace of raw per-feature variances = 30.000


## How much variance do the top components carry?

The explained variance ratio $\sum_{j=1}^{k} \lambda_j / \sum_{j=1}^{n} \lambda_j$
tells us how much information we keep when we compress 30 features into $k$.

In [5]:
explained = eigenvalues / eigenvalues.sum()
cumulative = np.cumsum(explained)

for k in [1, 2, 3, 5, 7, 10]:
    print(f"top-{k:2d} components: {cumulative[k-1]*100:6.2f}% of variance")

# Scree plot: individual + cumulative explained variance
fig = make_subplots(rows=1, cols=2, subplot_titles=('Explained variance per component', 'Cumulative explained variance'))
fig.add_trace(go.Bar(x=list(range(1, 31)), y=explained[:30], name='Per component',
                     marker_color=SITE['olive']), row=1, col=1)
fig.add_trace(go.Scatter(x=list(range(1, 31)), y=cumulative[:30], name='Cumulative',
                         line=dict(color=SITE['accent'], width=2)), row=1, col=2)
fig.add_hline(y=0.95, line_dash='dot', line_color=SITE['text_secondary'],
              annotation_text='95%', row=1, col=2)
fig.update_layout(
    title=dict(text='Explained variance of the breast cancer dataset', font=dict(color=SITE['text_primary'])),
    paper_bgcolor=SITE['bg_secondary'],
    plot_bgcolor=SITE['bg_primary'],
    font=dict(color=SITE['text_primary']),
    showlegend=False,
    autosize=True,
    margin=dict(l=60, r=40, t=80, b=60),
)
for axis in ['xaxis', 'yaxis', 'xaxis2', 'yaxis2']:
    fig.update_layout({axis: dict(gridcolor=SITE['border'], color=SITE['text_secondary'])})
fig.write_html(PLOTS_DIR / 'pca_explained_variance.html',
               config=dict(responsive=True, displayModeBar=True), include_plotlyjs='cdn')
fig.show()

top- 1 components:  43.88% of variance
top- 2 components:  62.92% of variance
top- 3 components:  72.53% of variance
top- 5 components:  84.70% of variance
top- 7 components:  90.91% of variance
top-10 components:  95.11% of variance


## Project the data onto the top-2 principal components

This is the same projection used to visualize the logistic regression decision
boundary in the previous post. Here we look at the data alone - do the two tumor
classes separate in the 2-D view?

In [6]:
P = Vt[:2].T                      # (30, 2) projection matrix: top-2 eigenvectors
X2d = centered @ P                # data in 2-D

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=X2d[y_all == 0, 0], y=X2d[y_all == 0, 1],
    mode="markers", name="Benign",
    marker=dict(color="#81b29a", size=6, opacity=0.8,
                line=dict(color=SITE['bg_secondary'], width=1)),
))
fig.add_trace(go.Scatter(
    x=X2d[y_all == 1, 0], y=X2d[y_all == 1, 1],
    mode="markers", name="Malignant",
    marker=dict(color="#e07a5f", size=6, opacity=0.8,
                line=dict(color=SITE['bg_secondary'], width=1)),
))
fig.update_layout(
    title=dict(text="Breast cancer data in PCA space (top-2 principal components)",
               font=dict(color=SITE['text_primary'])),
    xaxis=dict(title="PC 1", color=SITE['text_secondary'], gridcolor=SITE['border']),
    yaxis=dict(title="PC 2", color=SITE['text_secondary'], gridcolor=SITE['border']),
    paper_bgcolor=SITE['bg_secondary'],
    plot_bgcolor=SITE['bg_primary'],
    font=dict(color=SITE['text_primary']),
    legend=dict(orientation="h", x=0.5, y=-0.15, xanchor="center", yanchor="top",
                bgcolor=SITE['bg_secondary'], bordercolor=SITE['border']),
    autosize=True,
    margin=dict(l=60, r=40, t=60, b=60),
)
fig.write_html(PLOTS_DIR / 'pca_projection_2d.html',
               config=dict(responsive=True, displayModeBar=True), include_plotlyjs='cdn')
fig.show()

## Which features drive each component?

Each principal component is a linear combination of the original 30 features.
The loadings (eigenvector entries) tell us which features contribute most to
each direction - this is how PCA helps us *reason* about the data.

In [11]:
# Strip the norm_ prefix for readable labels
feature_names = [c.replace('norm_', '') for c in norm_cols]

loadings = pd.DataFrame({'feature': feature_names, 'PC1': P[:, 0], 'PC2': P[:, 1]})
loadings['abs_PC1'] = loadings['PC1'].abs()
loadings['abs_PC2'] = loadings['PC2'].abs()
loadings = loadings.sort_values('abs_PC1', ascending=False)

print("Top 10 features by |loading| on PC 1:")
print(loadings[['feature', 'PC1']].head(10).to_string(index=False))

fig = make_subplots(rows=1, cols=2, subplot_titles=('PC 1 loadings', 'PC 2 loadings'),
                    horizontal_spacing=0.25)
top10 = loadings.head(10)
pc2_top10 = loadings.sort_values('abs_PC2', ascending=False).head(10)
fig.add_trace(go.Bar(y=top10['feature'][::-1], x=top10['PC1'][::-1], orientation='h',
                     marker_color=SITE['accent'], name='PC1'), row=1, col=1)
fig.add_trace(go.Bar(y=pc2_top10['feature'][::-1], x=pc2_top10['PC2'][::-1],
                     orientation='h', marker_color=SITE['olive'], name='PC2'), row=1, col=2)
fig.update_layout(
    title=dict(text='Feature loadings on the top-2 principal components', font=dict(color=SITE['text_primary'])),
    paper_bgcolor=SITE['bg_secondary'],
    plot_bgcolor=SITE['bg_primary'],
    font=dict(color=SITE['text_primary']),
    showlegend=False,
    autosize=True,
    margin=dict(l=140, r=40, t=80, b=60),
)
for axis in ['xaxis', 'yaxis', 'xaxis2', 'yaxis2']:
    fig.update_layout({axis: dict(gridcolor=SITE['border'], color=SITE['text_secondary'])})
fig.write_html(PLOTS_DIR / 'pca_loadings.html',
               config=dict(responsive=True, displayModeBar=True), include_plotlyjs='cdn')
fig.show()

Top 10 features by |loading| on PC 1:
             feature      PC1
 concave points_mean 0.261832
      concavity_mean 0.259138
concave points_worst 0.250592
    compactness_mean 0.239593
     perimeter_worst 0.236742
        radius_worst 0.228658
     concavity_worst 0.228647
      perimeter_mean 0.228408
          area_worst 0.225220
           area_mean 0.222121


## Sanity check: eigenvectors of $\Sigma$ directly

To confirm the SVD shortcut, we form the covariance matrix explicitly and
eigendecompose it. The eigenvalues should match $\lambda_j = S_j^2/n$ from above.

In [9]:
Sigma = centered.T @ centered / len(centered)
eigvals_direct, eigvecs_direct = np.linalg.eigh(Sigma)  # eigh: for symmetric matrices

# eigh returns ascending order - flip to descending
eigvals_direct = eigvals_direct[::-1]
eigvecs_direct = eigvecs_direct[:, ::-1]

print("Max |difference| between SVD-derived and direct eigenvalues:",
      np.abs(eigenvalues - eigvals_direct).max())
print("Max |difference| between SVD-derived and direct eigenvectors (up to sign):",
      np.abs(np.abs(Vt) - np.abs(eigvecs_direct.T)).max())

Max |difference| between SVD-derived and direct eigenvalues: 3.9968028886505635e-15
Max |difference| between SVD-derived and direct eigenvectors (up to sign): 8.739189927275959e-13
